It becomes hard to individually apply different encoding techniques to different variables, and joining them becomes challenging. To handle this issue, Scikit-learn has a **ColumnTransformer**.

lets do one problem without using columnTransformr and with ColumnTransformer

In [1]:
#importing libraries
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [2]:
#importing dataset
df = pd.read_csv('covid_toy.csv')
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [3]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

fever has 10 missing value

- here age is discrete column,no preprocessing technique needs to applied for now (still can do scaling)
- gender is nominal so OneHotEncoding
- fever has missing value so imputer
- cough is ordinal data so ordinal encoding
- city is nominal so so OneHotEncoding
- has_covid is target column so LabelEncoder

In [4]:
#spilliting into training and testing
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2)

In [5]:
X_train

,age,gender,fever,cough,city
28,16,Male,104.0,Mild,Kolkata
15,70,Male,103.0,Strong,Kolkata
94,79,Male,NaN,Strong,Kolkata
80,14,Female,99.0,Mild,Mumbai
58,23,Male,98.0,Strong,Mumbai
...,...,...,...,...,...
9,64,Female,101.0,Mild,Delhi
62,56,Female,104.0,Strong,Bangalore
40,49,Female,102.0,Mild,Delhi
24,13,Female,100.0,Strong,Kolkata


### WITHOUT USING COLUMN TRANSFORMER

In [6]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])
                                 
X_train_fever.shape

(80, 1)

In [7]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [10]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [11]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [12]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

In [13]:
X_train_transformed

array([[ 16.        , 104.        ,   1.        ,   0.        ,
          1.        ,   0.        ,   0.        ],
       [ 70.        , 103.        ,   1.        ,   0.        ,
          1.        ,   0.        ,   1.        ],
       [ 79.        , 100.77333333,   1.        ,   0.        ,
          1.        ,   0.        ,   1.        ],
       [ 14.        ,  99.        ,   0.        ,   0.        ,
          0.        ,   1.        ,   0.        ],
       [ 23.        ,  98.        ,   1.        ,   0.        ,
          0.        ,   1.        ,   1.        ],
       [ 27.        , 100.        ,   1.        ,   0.        ,
          1.        ,   0.        ,   0.        ],
       [ 83.        ,  98.        ,   1.        ,   1.        ,
          0.        ,   0.        ,   0.        ],
       [ 65.        ,  99.        ,   1.        ,   1.        ,
          0.        ,   0.        ,   0.        ],
       [ 19.        , 100.        ,   0.        ,   0.        ,
          0.    

### USING COLUMN TRANSFORMER

In [14]:
from sklearn.compose import ColumnTransformer

In [16]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [17]:
transformer.fit_transform(X_train).shape

(80, 7)

In [18]:
transformer.transform(X_test).shape

(20, 7)